In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
from urllib.parse import urljoin
import os
import csv

# ================= CẤU HÌNH CRAWLER =================
START_YEAR = 2023
END_YEAR = 2025
OUTPUT_DIR = "vietnamnet_data_csv"

# Toàn bộ danh mục Vietnamnet (Dữ liệu bạn cung cấp)

CATEGORIES  = {
        "thoi su": {
        "dan sinh": "https://vietnamnet.vn/thoi-su/dan-sinh",
        "giao thong": "https://vietnamnet.vn/thoi-su/giao-thong",
        "tin nong": "https://vietnamnet.vn/thoi-su/tin-nong",
        "đo thi": "https://vietnamnet.vn/thoi-su/do-thi"
    },
    "kinh doanh": {
        "net zero": "https://vietnamnet.vn/net-zero",
        "tai chinh": "https://vietnamnet.vn/kinh-doanh/tai-chinh",
        "đau tu": "https://vietnamnet.vn/kinh-doanh/dau-tu",
        "thi truong": "https://vietnamnet.vn/kinh-doanh/thi-truong",
        "doanh nhan": "https://vietnamnet.vn/kinh-doanh/doanh-nhan",
        "tu van tai chinh": "https://vietnamnet.vn/kinh-doanh/tu-van-tai-chinh",
        "điem tin dung cong dan 4.0": "https://vietnamnet.vn/kinh-doanh/diem-tin-dung-cong-dan-4-0"
    },
    "giao duc": {
        "nha truong": "https://vietnamnet.vn/giao-duc/nha-truong",
        "chan dung": "https://vietnamnet.vn/giao-duc/chan-dung",
        "goc phu huynh": "https://vietnamnet.vn/giao-duc/goc-phu-huynh",
        "tuyen sinh": "https://vietnamnet.vn/giao-duc/tuyen-sinh",
        "du hoc": "https://vietnamnet.vn/giao-duc/du-hoc",
    },
    "the gioi": {
        "binh luan quoc te": "https://vietnamnet.vn/the-gioi/binh-luan-quoc-te",
        "chan dung": "https://vietnamnet.vn/the-gioi/chan-dung",
        "ho so": "https://vietnamnet.vn/the-gioi/ho-so",
        "the gioi đo đay": "https://vietnamnet.vn/the-gioi/the-gioi-do-day",
        "viet nam va the gioi": "https://vietnamnet.vn/the-gioi/viet-nam-va-the-gioi",
        "quan su": "https://vietnamnet.vn/the-gioi/quan-su"
    },
    "the thao": {
        "bong đa viet nam": "https://vietnamnet.vn/the-thao/bong-da-viet-nam",
        "bong đa quoc te": "https://vietnamnet.vn/the-thao/bong-da-quoc-te",
        "hau truong": "https://vietnamnet.vn/the-thao/hau-truong",
        "cac mon khac": "https://vietnamnet.vn/the-thao/cac-mon-khac"
    },
    "đoi song": {
        "gia đinh": "https://vietnamnet.vn/doi-song/gia-dinh",
        "chuyen la": "https://vietnamnet.vn/doi-song/chuyen-la",
        "am thuc": "https://vietnamnet.vn/doi-song/am-thuc",
        "gioi tre": "https://vietnamnet.vn/doi-song/gioi-tre",
        "meo vat": "https://vietnamnet.vn/doi-song/meo-vat",
    },
    "suc khoe": {
        "tin tuc": "https://vietnamnet.vn/suc-khoe/suc-khoe-24h",
        "lam đep": "https://vietnamnet.vn/suc-khoe/lam-dep",
        "tu van suc khoe": "https://vietnamnet.vn/suc-khoe/tu-van-suc-khoe",
        "đan ong": "https://vietnamnet.vn/suc-khoe/dan-ong",
        "cac loai benh": "https://vietnamnet.vn/suc-khoe/benh"
    },
    "cong nghe": {
        "thi truong": "https://vietnamnet.vn/cong-nghe/thi-truong",
        "chuyen đoi so": "https://vietnamnet.vn/cong-nghe/chuyen-doi-so",
        "ha tang so": "https://vietnamnet.vn/cong-nghe/ha-tang-so",
        "an ninh mang": "https://vietnamnet.vn/cong-nghe/an-ninh-mang",
        "san pham": "https://vietnamnet.vn/cong-nghe/san-pham",
        "ai": "https://vietnamnet.vn/cong-nghe/ai"
    },
    "phap luat": {
        "ho so vu an": "https://vietnamnet.vn/phap-luat/ho-so-vu-an",
        "tu van phap luat": "https://vietnamnet.vn/phap-luat/tu-van-phap-luat",
        "ky su phap đinh": "https://vietnamnet.vn/phap-luat/ky-su-phap-dinh"
    },
    "xe": {
        "xe moi": "https://vietnamnet.vn/oto-xe-may/xe-moi",
        "kham pha": "https://vietnamnet.vn/oto-xe-may/kham-pha",
        "sau tay lai": "https://vietnamnet.vn/oto-xe-may/sau-tay-lai",
        "dien đan": "https://vietnamnet.vn/oto-xe-may/dien-dan",
        "đanh gia xe": "https://vietnamnet.vn/oto-xe-may/danh-gia-xe",
        "gia xe": "https://vietnamnet.vn/oto-xe-may/gia-xe",
        "tu van": "https://vietnamnet.vn/oto-xe-may/tu-van",
        "du lieu xe": "https://vietnamnet.vn/oto-xe-may/du-lieu-xe",
        "video xe": "https://vietnamnet.vn/video/oto-xe-may"
    },
    "bat đong san": {
        "du an": "https://vietnamnet.vn/bat-dong-san/du-an",
        "noi that": "https://vietnamnet.vn/bat-dong-san/noi-that",
        "tu van": "https://vietnamnet.vn/bat-dong-san/kinh-nghiem-tu-van",
        "thi truong": "https://vietnamnet.vn/bat-dong-san/thi-truong",
        "nha đep": "https://vietnamnet.vn/bat-dong-san/nha-dep",
        "co hoi an cu": "https://vietnamnet.vn/bat-dong-san/kim-oanh-group"
    },
    "du lich": {
        "chuyen cua nhung dong song": "https://vietnamnet.vn/du-lich/chuyen-cua-nhung-dong-song-sk0008F1.html",
        "đi đau choi đi": "https://vietnamnet.vn/du-lich/di-dau-choi-di",
        "an an uong uong": "https://vietnamnet.vn/du-lich/an-uong",
        "ngu ngu nghi nghi": "https://vietnamnet.vn/du-lich/ngu-nghi"
    }
}


BASE_DOMAIN = "https://vietnamnet.vn"

def build_page_url(base_url, page_num):
    """Xây dựng URL phân trang theo format của Vietnamnet"""
    if page_num == 1:
        return base_url
    
    # Xử lý các URL đặc biệt có đuôi .html
    if base_url.endswith('.html'):
        return base_url.replace('.html', f'-page{page_num}.html')
    else:
        # Format chuẩn: /chuyen-muc-page2
        base_url = base_url.rstrip('/')
        return f"{base_url}-page{page_num}"

def get_article_links(category_url):
    """Lấy danh sách link bài viết trên trang chuyên mục"""
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0"}
    try:
        response = requests.get(category_url, headers=headers, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        links = []
        
        # Vietnamnet thường đặt link trong h3.vnn-title hoặc h3.title-bold
        article_tags = soup.select('h3.vnn-title a, h3.title-bold a, .vnn-title a')
        
        for a_tag in article_tags:
            href = a_tag.get('href')
            if href:
                link = href if href.startswith('http') else urljoin(BASE_DOMAIN, href)
                # Loại bỏ link trùng lặp để giữ đúng thứ tự bài mới -> cũ
                if link not in links and '.html' in link:
                    links.append(link)
                        
        return links
    except Exception as e:
        print(f"Lỗi khi lấy link từ {category_url}: {e}")
        return []

def parse_article(url, category, sub_topic):
    """Trích xuất dữ liệu chi tiết an toàn chống lỗi NoneType và bao phủ toàn bộ JSON Schema"""
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0"}
    article_data = {
        "category": category, "sub_topic": sub_topic, "title": "",
        "description": "", "content": "", "public_date": "",
        "author": "", "source": "Vietnamnet", "url": url, "tag": []
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # --- FIX 1: Xử lý linh hoạt mảng JSON-LD để không bị lọt bài báo ---
        def extract_schema(item):
            if isinstance(item, dict) and item.get("@type") in ["NewsArticle", "Article"]:
                article_data["title"] = item.get("headline", "")
                article_data["description"] = item.get("description", "")
                article_data["public_date"] = item.get("datePublished", "")
                
                author_info = item.get("author", {})
                if isinstance(author_info, dict):
                    article_data["author"] = author_info.get("name", "")
                elif isinstance(author_info, list) and len(author_info) > 0:
                    first_author = author_info[0]
                    if isinstance(first_author, dict):
                        article_data["author"] = first_author.get("name", "")
                    else:
                        article_data["author"] = str(first_author)

        for script in soup.find_all('script', type='application/ld+json'):
            try:
                raw_json = script.string if script.string else script.text
                if not raw_json: continue
                data = json.loads(raw_json)
                
                # Quét trường hợp Vietnamnet nhét nhiều dict vào một list
                if isinstance(data, dict):
                    extract_schema(data)
                elif isinstance(data, list):
                    for element in data:
                        extract_schema(element)
            except Exception:
                continue
        
        # Lấy tag an toàn
        keywords_meta = soup.find('meta', {'name': 'keywords'})
        if keywords_meta and keywords_meta.has_attr('content'):
            article_data["tag"] = [tag.strip() for tag in keywords_meta['content'].split(',')]
                
        # --- FIX 2: Bóc tách nội dung an toàn chống lỗi NoneType ---
        content_div = soup.find('div', class_='maincontent') 
        
        if content_div:
            for unwanted in content_div(['script', 'style', 'figure', 'iframe', 'table', 'div']):
                # QUAN TRỌNG: Nếu thẻ cha đã bị xóa trước đó, thẻ con sẽ bị mất thuộc tính (parent = None).
                # Lệnh continue này chặn việc gọi hàm .get() vào vùng nhớ None.
                if unwanted.parent is None:
                    continue
                    
                if unwanted.name in ['script', 'style', 'figure', 'iframe', 'table']:
                    unwanted.decompose()
                elif unwanted.name == 'div':
                    classes = unwanted.get('class')
                    # Đảm bảo classes là một mảng trước khi dùng toán tử 'in'
                    if isinstance(classes, list) and 'vnn-box-video' in classes:
                        unwanted.decompose()
            
            paragraphs = content_div.find_all(['p', 'h1', 'h2', 'h3'])
            text_lines = [p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)]
            article_data["content"] = "\n".join(text_lines)
            
    except Exception as e:
        print(f"Lỗi khi parse {url}: {e}")
        
    return article_data

def extract_year(iso_date_str):
    """Trích xuất năm từ chuỗi thời gian (vd: 2024-05-10T14:30:00+07:00)"""
    if not iso_date_str:
        return None
    try:
        return int(iso_date_str[:4])
    except ValueError:
        return None

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    fieldnames = ["category", "sub_topic", "title", "description", "content", "public_date", "author", "source", "url", "tag"]
    
    for category, sub_topics in CATEGORIES.items():
        safe_category_name = category.replace(" ", "_").replace("&", "va")
        csv_file_path = os.path.join(OUTPUT_DIR, f"{safe_category_name}.csv")
        file_exists = os.path.exists(csv_file_path)
        
        # Dùng utf-8-sig để Excel trên Windows hiển thị tiếng Việt không bị lỗi font
        with open(csv_file_path, 'a', newline='', encoding='utf-8-sig') as f_out:
            writer = csv.DictWriter(f_out, fieldnames=fieldnames)
            if not file_exists:
                writer.writeheader()
                
            for sub_topic, base_url in sub_topics.items():
                print(f"\n--- Đang cào: {category} > {sub_topic} ---")
                page = 1
                stop_category = False
                
                # Bộ đếm an toàn: Gặp 3 bài cũ liên tiếp mới cho phép thoát
                consecutive_old_articles = 0 
                
                while not stop_category:
                    page_url = build_page_url(base_url, page)
                    print(f"  > Trang {page}: {page_url}")
                    
                    article_links = get_article_links(page_url)
                    if not article_links:
                        print("  => Không còn bài viết hoặc hết trang. Chuyển sub-topic.")
                        break 
                    
                    for link in article_links:
                        data = parse_article(link, category, sub_topic)
                        year = extract_year(data['public_date'])
                        
                        if not year:
                            continue
                            
                        # KIỂM TRA MỐC THỜI GIAN
                        if year > END_YEAR:
                            consecutive_old_articles = 0 # Bài mới, bỏ qua nhưng reset đếm
                            continue
                            
                        elif START_YEAR <= year <= END_YEAR:
                            consecutive_old_articles = 0 # Đúng bài, reset đếm
                            if isinstance(data['tag'], list):
                                data['tag'] = ", ".join(data['tag'])
                            writer.writerow(data)
                            
                        elif year < START_YEAR:
                            consecutive_old_articles += 1
                            print(f"  => Vấp bài báo năm {year} (Lần {consecutive_old_articles}/3)")
                            if consecutive_old_articles >= 3:
                                print(f"  => Đã qua vùng thời gian an toàn. NGẮT vòng lặp.")
                                stop_category = True
                                break 
                        
                        # Giảm tải cho server
                        time.sleep(0.5) 
                        
                    page += 1

if __name__ == "__main__":
    main()


--- Đang cào: the thao > tin chuyen nhuong ---
  > Trang 1: https://vietnamnet.vn/the-thao/tin-chuyen-nhuong
  > Trang 2: https://vietnamnet.vn/the-thao/tin-chuyen-nhuong-page2
  > Trang 3: https://vietnamnet.vn/the-thao/tin-chuyen-nhuong-page3
  > Trang 4: https://vietnamnet.vn/the-thao/tin-chuyen-nhuong-page4
  > Trang 5: https://vietnamnet.vn/the-thao/tin-chuyen-nhuong-page5
  > Trang 6: https://vietnamnet.vn/the-thao/tin-chuyen-nhuong-page6
  > Trang 7: https://vietnamnet.vn/the-thao/tin-chuyen-nhuong-page7
  > Trang 8: https://vietnamnet.vn/the-thao/tin-chuyen-nhuong-page8
  > Trang 9: https://vietnamnet.vn/the-thao/tin-chuyen-nhuong-page9
  > Trang 10: https://vietnamnet.vn/the-thao/tin-chuyen-nhuong-page10
  > Trang 11: https://vietnamnet.vn/the-thao/tin-chuyen-nhuong-page11
  > Trang 12: https://vietnamnet.vn/the-thao/tin-chuyen-nhuong-page12
  > Trang 13: https://vietnamnet.vn/the-thao/tin-chuyen-nhuong-page13
  > Trang 14: https://vietnamnet.vn/the-thao/tin-chuyen-nhuong-pag